### Next word prediction Using LSTM

In [1]:
faqs = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don’t have to wait for a month to end.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""

In [2]:
import tensorflow as tf
from keras.layers import TextVectorization
from keras.utils import pad_sequences, to_categorical
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
import numpy as np
import time

#### 1. Input Data Creation

In [3]:
vectorizer = TextVectorization(output_mode='int')

In [4]:
vectorizer.adapt([faqs])

In [5]:
vocab_size = len(vectorizer.get_vocabulary())
print(vocab_size)

276


In [6]:
input_sequences = []
for sentence in faqs.split('\n'):
  tokenized_sentence = vectorizer([sentence]).numpy()[0]

  for i in range(1,len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])

In [7]:
input_sequences[:10]

[array([137,   2]),
 array([137,   2,  14]),
 array([12,  8]),
 array([12,  8,  2]),
 array([12,  8,  2, 13]),
 array([12,  8,  2, 13, 48]),
 array([12,  8,  2, 13, 48, 15]),
 array([12,  8,  2, 13, 48, 15, 57]),
 array([12,  8,  2, 13, 48, 15, 57, 68]),
 array([ 12,   8,   2,  13,  48,  15,  57,  68, 113])]

In [8]:
max_len = max([len(x) for x in input_sequences])
print(max_len)

57


In [9]:
padded_input_sequences = pad_sequences(
    input_sequences, 
    maxlen = max_len, 
    padding='pre'
)

In [10]:
X = padded_input_sequences[:,:-1]
y = padded_input_sequences[:,-1]

In [11]:
print(X.shape)
print(y.shape)

(824, 56)
(824,)


In [12]:
y = to_categorical(y,num_classes=vocab_size)

In [13]:
print(y.shape)

(824, 276)


#### 2. Define the Model architecture

In [14]:
embedding_dim = 100

In [15]:
layer1 = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim
)

layer2 = LSTM(150)
layer3 = Dense(vocab_size, activation='softmax')
model = Sequential([layer1, layer2, layer3])
model.compile(
    loss='categorical_crossentropy', 
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.fit(X,y, epochs=100)

Epoch 1/100


26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.0667 - loss: 5.4257
Epoch 2/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.0813 - loss: 5.0391
Epoch 3/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.0813 - loss: 4.9625
Epoch 4/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.0813 - loss: 4.9301
Epoch 5/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0813 - loss: 4.8565
Epoch 6/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0971 - loss: 4.7592
Epoch 7/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1214 - loss: 4.6008
Epoch 8/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.1359 - loss: 4.3961
Epoch 9/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.1784 - loss: 4.2029
Epoch 10/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.1978 - loss: 3.9842
Epoch 11/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2172 - loss: 3.7743
Epoch 12/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy:

In [ ]:
vocabulary = vectorizer.get_vocabulary()
# vocabulary[0] = '' (padding), vocabulary[1] = '[UNK]', vocabulary[2+] = actual words

def predict_next_word(text):
    token_text = vectorizer([text]).numpy()[0]          # tokenize → 1D array
    padded = pad_sequences([token_text], maxlen=56, padding='pre')  # pad to training length
    pos = np.argmax(model.predict(padded, verbose=0))   # predicted index
    return vocabulary[pos]   

In [ ]:
text = "what are the"

for _ in range(10):
    next_word = predict_next_word(text)
    text = text + " " + next_word
    print(text)
    time.sleep(2)
